# Eval Snapshots

For each evaluation checkpoint, shows predicted keypoints on the same fixed tiles — a filmstrip of how detections evolve during training.

In [ ]:
import sys
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

sys.path.append(str(Path.cwd().parent.parent))
sys.path.append(str(Path.cwd().parent))

import model_instance as _mi_mod
importlib.reload(_mi_mod)
from model_instance import ModelInstance, TrainingConfig

In [ ]:
RUN_NAME    = "oyster"
RUNS_SOURCE = "git_saved_runs"   # "runs" | "git_saved_runs"
TILE_INDICES = [0, 1, 2, 3]     # which of the 32 snapshot tiles to show (rows)
SIDE        = "he"              # "he" | "ihc" | "both"
KP_COLOR    = "#facc15"
KP_SIZE     = 4

In [ ]:
_REPO_ROOT = Path.cwd().parent.parent

if RUNS_SOURCE == "git_saved_runs":
    log_path = _REPO_ROOT / "introducing_superpoint" / "git_saved_runs" / "training_log.json"
else:
    log_path = ModelInstance(name=RUN_NAME, config=TrainingConfig(name=RUN_NAME)).log_path

instance = ModelInstance.load_log(log_path)
evals_with_snaps = [e for e in instance.evaluation_logs if e.tile_snapshots]
print(f"{len(evals_with_snaps)} eval checkpoints with snapshots")
if evals_with_snaps:
    print(f"tiles per snapshot: {len(evals_with_snaps[0].tile_snapshots)}")
    print(f"samples_seen range: {evals_with_snaps[0].samples_seen} – {evals_with_snaps[-1].samples_seen}")

In [ ]:
def load_img(tile_dir, side):
    return np.array(Image.open(Path(tile_dir) / f"{side}.png").convert("L"))


def render_snap(ax, tile_dir, keypoints, side_label, samples_seen):
    img = load_img(tile_dir, side_label)
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    if keypoints:
        xy = np.array(keypoints)
        ax.scatter(xy[:, 0], xy[:, 1], s=KP_SIZE, c=KP_COLOR, linewidths=0)
    ax.set_title(f"{samples_seen // 1000}K  n={len(keypoints)}", fontsize=7)
    ax.axis("off")


sides = ["he", "ihc"] if SIDE == "both" else [SIDE]
evals_sorted = sorted(evals_with_snaps, key=lambda e: e.samples_seen)
n_cols = len(evals_sorted)
n_rows = len(TILE_INDICES) * len(sides)

if not evals_sorted:
    print("No snapshots found. Run training with eval_snapshot_tiles > 0 first.")
else:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes[np.newaxis, :]
    elif n_cols == 1:
        axes = axes[:, np.newaxis]

    for row_idx, (tile_idx, side) in enumerate([
        (ti, s) for ti in TILE_INDICES for s in sides
    ]):
        for col_idx, ev in enumerate(evals_sorted):
            ax = axes[row_idx, col_idx]
            snaps = ev.tile_snapshots
            if tile_idx >= len(snaps):
                ax.axis("off")
                continue
            snap = snaps[tile_idx]
            kps  = snap[f"{side}_keypoints"]
            render_snap(ax, snap["tile_dir"], kps, side, ev.samples_seen)
            if col_idx == 0:
                ax.set_ylabel(f"tile {tile_idx} {side}", fontsize=7)

    fig.suptitle(
        f"{instance.name} — keypoint evolution across {n_cols} eval checkpoints",
        fontsize=10,
    )
    fig.tight_layout()
    plt.show()